In [1]:
import nidaqmx
import time

with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan("Dev1/ai0")  # change ai0 to your actual channel
    task.timing.cfg_samp_clk_timing(rate=2500, sample_mode=nidaqmx.constants.AcquisitionType.CONTINUOUS)
    task.start()

    print("Reading 2 seconds of live data...")
    start = time.time()
    while time.time() - start < 2:
        data = task.read(number_of_samples_per_channel=250)
        print(f"min={min(data):.3f}  max={max(data):.3f}  mean={sum(data)/len(data):.3f}")

Reading 2 seconds of live data...
min=1.243  max=1.400  mean=1.309
min=1.162  max=1.243  mean=1.203
min=1.098  max=1.161  mean=1.132
min=1.044  max=1.099  mean=1.073
min=0.996  max=1.045  mean=1.023
min=0.954  max=0.999  mean=0.978
min=0.915  max=0.957  mean=0.937
min=0.879  max=0.919  mean=0.900
min=0.845  max=0.883  mean=0.865
min=0.814  max=0.849  mean=0.832
min=0.784  max=0.818  mean=0.802
min=0.756  max=0.789  mean=0.773
min=0.730  max=0.761  mean=0.746
min=0.704  max=0.735  mean=0.720
min=0.680  max=0.710  mean=0.695
min=0.657  max=0.686  mean=0.672
min=0.636  max=0.663  mean=0.650
min=0.615  max=0.642  mean=0.628
min=0.594  max=0.621  mean=0.608
min=0.575  max=0.600  mean=0.588


In [4]:
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import threading, queue, time
import numpy as np
import pandas as pd
from scipy import signal as sg

FS = 2500
CHUNK = 100  # samples per chunk, 40ms

raw_data = pd.read_excel('test1.xlsx', engine='calamine')['Voltage (Collected)'].values.astype(float)

emg_queue = queue.Queue(maxsize=50)

def replay_thread():
    idx = 0
    period = CHUNK / FS
    next_t = time.perf_counter()
    while idx < len(raw_data):
        chunk = raw_data[idx:idx+CHUNK]
        idx += CHUNK
        try:
            emg_queue.put_nowait(chunk)
        except queue.Full:
            pass
        next_t += period
        sleep_for = next_t - time.perf_counter()
        if sleep_for > 0:
            time.sleep(sleep_for)

t = threading.Thread(target=replay_thread, daemon=True)
t.start()

# quick check: drain queue for 3 seconds, print chunk count
start = time.time()
count = 0
while time.time() - start < 3:
    if not emg_queue.empty():
        emg_queue.get()
        count += 1
print(f"Chunks received in 3s: {count}  (expected ~{int(3*FS/CHUNK)})")

Chunks received in 3s: 75  (expected ~75)


In [11]:
nyq = FS / 2
bp_sos = sg.butter(4, [20/nyq, 450/nyq], btype='bandpass', output='sos')
notch_b, notch_a = sg.iirnotch(50/nyq, 30)

result_queue = queue.Queue(maxsize=50)

def processing_thread():
    bp_zi = sg.sosfilt_zi(bp_sos)
    notch_zi = sg.lfilter_zi(notch_b, notch_a)
    buffer = np.empty(0)
    WIN = int(FS * 0.2)   # shorter window (200ms) for snappier live feedback

    while True:
        try:
            chunk = emg_queue.get(timeout=1)
        except queue.Empty:
            break

        filtered, bp_zi = sg.sosfilt(bp_sos, chunk, zi=bp_zi)
        filtered, notch_zi = sg.lfilter(notch_b, notch_a, filtered, zi=notch_zi)
        buffer = np.concatenate([buffer, filtered])

        if len(buffer) >= WIN:
            w = buffer[:WIN]
            buffer = buffer[WIN:]
            rms = float(np.sqrt(np.mean(w**2)))
            try:
                result_queue.put_nowait(rms)
            except queue.Full:
                pass

# restart a fresh replay + processing pair
emg_queue2 = queue.Queue(maxsize=50)

def replay_thread2():
    idx = 0
    period = CHUNK / FS
    next_t = time.perf_counter()
    while idx < len(raw_data):
        chunk = raw_data[idx:idx+CHUNK]
        idx += CHUNK
        try:
            emg_queue2.put_nowait(chunk)
        except queue.Full:
            pass
        next_t += period
        sleep_for = next_t - time.perf_counter()
        if sleep_for > 0:
            time.sleep(sleep_for)

# rewire processing_thread to use emg_queue2
def processing_thread2():
    bp_zi = sg.sosfilt_zi(bp_sos)
    notch_zi = sg.lfilter_zi(notch_b, notch_a)
    buffer = np.empty(0)
    WIN = int(FS * 0.2)

    while True:
        try:
            chunk = emg_queue2.get(timeout=1)
        except queue.Empty:
            break
        filtered, bp_zi = sg.sosfilt(bp_sos, chunk, zi=bp_zi)
        filtered, notch_zi = sg.lfilter(notch_b, notch_a, filtered, zi=notch_zi)
        buffer = np.concatenate([buffer, filtered])
        if len(buffer) >= WIN:
            w = buffer[:WIN]
            buffer = buffer[WIN:]
            rms = float(np.sqrt(np.mean(w**2)))
            try:
                result_queue.put_nowait(rms)
            except queue.Full:
                pass

threading.Thread(target=replay_thread2, daemon=True).start()
threading.Thread(target=processing_thread2, daemon=True).start()

start = time.time()
while time.time() - start < 5:
    if not result_queue.empty():
        rms = result_queue.get()
        print(f"t={time.time()-start:.1f}s  RMS={rms:.3f}")
    time.sleep(0.05)

t=0.2s  RMS=0.113
t=0.4s  RMS=0.024
t=0.6s  RMS=0.026
t=0.8s  RMS=0.024
t=1.0s  RMS=0.030
t=1.2s  RMS=0.023
t=1.4s  RMS=0.026
t=1.6s  RMS=0.034
t=1.8s  RMS=0.189
t=2.0s  RMS=0.131
t=2.2s  RMS=0.180
t=2.4s  RMS=0.272
t=2.6s  RMS=0.405
t=2.8s  RMS=0.485
t=3.0s  RMS=0.913
t=3.2s  RMS=0.999
t=3.4s  RMS=1.104
t=3.6s  RMS=1.239
t=3.8s  RMS=1.612
t=4.0s  RMS=1.575
t=4.2s  RMS=1.934
t=4.4s  RMS=1.705
t=4.6s  RMS=1.161
t=4.8s  RMS=0.739
t=5.0s  RMS=0.542


In [18]:
import cv2
import mediapipe as mp

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)

OpenCV: 5.0.0
MediaPipe: 1.0.1


In [ ]:
import sys
print(sys.version)

import cv2
import mediapipe as mp
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("Real MediaPipe:", hasattr(mp, 'solutions'))

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
OpenCV: 5.0.0
MediaPipe: 1.0.1
Real MediaPipe: False


In [ ]:
import subprocess
result = subprocess.run(['pip', 'show', 'mediapipe'], capture_output=True, text=True)
print(result.stdout)

print(dir(mp))

Name: mediapipe
Version: 1.0.1
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web. See the privacy notice at https://goo.gle/mediapipe-privacy.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: D:\sudarshan\emg_vs\cv_env\Lib\site-packages
Requires: absl-py, certifi, flatbuffers, matplotlib, numpy, opencv-contrib-python, sounddevice
Required-by: 

['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [ ]:
import urllib.request

model_url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
urllib.request.urlretrieve(model_url, "pose_landmarker_lite.task")
print("Model downloaded")

Model downloaded


In [ ]:
import mediapipe as mp
print(mp.__file__)
print(hasattr(mp, 'solutions'))

c:\Users\Luffy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mediapipe\__init__.py
False


In [8]:
import cv2

cap = cv2.VideoCapture(0)
print("Camera opened:", cap.isOpened())

ret, frame = cap.read()
print("Frame read success:", ret)
if ret:
    print("Frame shape:", frame.shape)

cap.release()

Camera opened: True
Frame read success: True
Frame shape: (480, 640, 3)


In [ ]:
import cv2

cap = cv2.VideoCapture(0)

for i in range(100):  # run for a limited number of frames, not infinite
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow('Camera Test', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Done")

KeyboardInterrupt: 

In [12]:
import cv2
import time

cap = cv2.VideoCapture(0)
start = time.time()
frame_count = 0

while time.time() - start < 10:  # run for 10 real seconds
    ret, frame = cap.read()
    if not ret:
        print("Frame read failed")
        break
    frame_count += 1
    cv2.imshow('Camera Test', frame)
    cv2.waitKey(1)

cap.release()
cv2.destroyAllWindows()
print(f"Done. Frames shown: {frame_count}")

Done. Frames shown: 294


In [13]:
import cv2
info = cv2.getBuildInformation()
# print just the GUI-related section
for line in info.split('\n'):
    if 'GUI' in line or 'QT' in line or 'Win32' in line or 'highgui' in line.lower():
        print(line)

    To be built:                 bgsegm bioinspired calib ccalib core datasets dnn dnn_objdetect dnn_superres dpm face features flann fuzzy gapi geometry hfs highgui img_hash imgcodecs imgproc intensity_transform line_descriptor ml objdetect optflow phase_unwrapping photo plot ptcloud python3 quality rapid reg rgbd saliency shape signal stereo stitching structured_light superres surface_matching text tracking video videoio videostab wechat_qrcode xfeatures2d ximgproc xobjdetect xphoto xstereo
  GUI:                           WIN32UI
    Win32 UI:                    YES
